In [9]:
import pandas as pd
import numpy as np
from scipy.stats import norm
import os
import re
import calendar
from google.colab import drive
from tqdm.auto import tqdm

# --- 1. 強健掛載函數 ---
def mount_google_drive():
    """確保 Google Drive 正確掛載，處理斷線問題"""
    drive_path = '/content/drive'
    try:
        if not os.path.ismount(drive_path):
            print("⏳ 正在掛載 Google Drive...")
            drive.mount(drive_path, force_remount=True)
        else:
            # 測試是否真的能讀取（防止 Transport endpoint is not connected）
            os.listdir(drive_path)
            print("✅ Google Drive 已在線上。")
    except OSError:
        print("⚠️ 偵測到掛載點異常，正在強制重新連線...")
        # 解除掛載並重新掛載
        os.system(f"fusermount -u {drive_path}")
        drive.mount(drive_path, force_remount=True)

# --- 2. IV 核心運算函數 ---
def bs_call_price(S, K, T, r, sigma):
    if sigma <= 0 or T <= 0: return 0
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

def bs_put_price(S, K, T, r, sigma):
    if sigma <= 0 or T <= 0: return 0
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

def find_iv(market_price, S, K, T, r, cp_flag='C'):
    if pd.isna(T) or T <= 0: return np.nan

    if 'C' in cp_flag:
        intrinsic = max(0, S - K * np.exp(-r * T))
        func = bs_call_price
    else:
        intrinsic = max(0, K * np.exp(-r * T) - S)
        func = bs_put_price

    if market_price <= intrinsic or market_price <= 0: return np.nan

    # 防線：設定隱含波動率上限
    low, high = 1e-5, 5.0
    for _ in range(40):
        mid = (low + high) / 2
        price = func(S, K, T, r, mid)
        if abs(price - market_price) < 1e-5: return mid
        if price < market_price: low = mid
        else: high = mid
    return mid

# --- 3. 輔助函數 ---
def get_expiry_date(contract_str):
    contract_str = str(contract_str).strip()
    if len(contract_str) < 6: return pd.NaT
    try:
        year = int(contract_str[:4])
        month = int(contract_str[4:6])
    except:
        return pd.NaT

    c = calendar.monthcalendar(year, month)
    wednesdays = [week[calendar.WEDNESDAY] for week in c if week[calendar.WEDNESDAY] != 0]

    w_num = 3
    if 'W' in contract_str.upper():
        try:
            w_str = contract_str.upper().split('W')[1]
            w_num = int(w_str[0])
        except:
            pass

    if w_num > len(wednesdays): w_num = len(wednesdays)
    day = wednesdays[w_num - 1]
    return pd.Timestamp(year, month, day)

def robust_read_csv(file_path):
    strategies = [
        {'enc': 'big5', 'sep': ','}, {'enc': 'utf-8-sig', 'sep': ','},
        {'enc': 'cp950', 'sep': ','}, {'enc': 'utf-8', 'sep': '\t'},
        {'enc': 'big5', 'sep': '\t'}, {'enc': 'utf-16', 'sep': '\t'}
    ]
    for strategy in strategies:
        try:
            df = pd.read_csv(file_path, encoding=strategy['enc'], sep=strategy['sep'],
                             dtype=str, on_bad_lines='skip', skipinitialspace=True)
            if len(df.columns) > 1: return df
        except Exception: continue
    return pd.DataFrame()

# --- 4. 主程式 ---
def main():
    # 執行掛載檢查
    mount_google_drive()

    base_path = '/content/drive/MyDrive/金融資料探勘'

    # 再次確認路徑是否存在，避免 OSError
    if not os.path.exists(base_path):
        print(f"❌ 找不到資料夾：{base_path}")
        print("💡 請確認該資料夾位於『我的雲端硬碟』中，而非僅在『與我共用』。")
        return

    # 僅保留 2023 年的資料夾
    target_folders = ['Option_2023_Clean']
    s0_files = ['2023_S0.csv']
    s0_dict = {}

    # 載入 S0 價格
    for s0_name in s0_files:
        s0_path = os.path.join(base_path, s0_name)
        if os.path.exists(s0_path):
            df_s0 = robust_read_csv(s0_path)
            if not df_s0.empty:
                df_s0.columns = df_s0.columns.str.replace(r'\s+', '', regex=True)
                if '年月日' in df_s0.columns and '收盤價(元)' in df_s0.columns:
                    df_s0['Date'] = pd.to_datetime(df_s0['年月日']).dt.strftime('%Y/%m/%d')
                    df_s0['S0'] = pd.to_numeric(df_s0['收盤價(元)'], errors='coerce')
                    s0_dict.update(dict(zip(df_s0['Date'], df_s0['S0'])))
                    print(f"✅ 成功載入標的價格對照表：{s0_name}")
        else:
            print(f"⚠️ 找不到 {s0_name}。")

    all_data_list = []

    for folder_name in target_folders:
        folder_path = os.path.join(base_path, folder_name)
        if not os.path.exists(folder_path):
            print(f"⚠️ 跳過不存在的資料夾：{folder_name}")
            continue

        csv_files = sorted([f for f in os.listdir(folder_path) if f.lower().endswith('.csv')])
        print(f"📂 正在處理資料夾：{folder_name} (共 {len(csv_files)} 個檔案)")

        pbar = tqdm(csv_files, desc=f"⏳ {folder_name} 進度", unit="天")
        for csv_name in pbar:
            pbar.set_postfix_str(f"處理中: {csv_name}")
            csv_path = os.path.join(folder_path, csv_name)

            date_match = re.search(r'(\d{4})_(\d{2})_(\d{2})', csv_name)
            if not date_match: continue

            yyyy, mm, dd = date_match.group(1), date_match.group(2), date_match.group(3)
            raw_date_str = f"{yyyy}/{mm}/{dd}"
            dt_object = pd.to_datetime(raw_date_str)

            # 設定無風險利率 Rf
            if dt_object <= pd.Timestamp('2023-03-26'): rf = 0.00975
            else: rf = 0.011

            s0 = s0_dict.get(dt_object.strftime('%Y/%m/%d'))
            if s0 is None: continue

            df = robust_read_csv(csv_path)
            if df.empty: continue

            df.columns = df.columns.str.replace(r'\s+', '', regex=True)
            mapping = {'商品代號': 'Symbol', '履約價格': 'StrikePrice', '成交價格': 'Price',
                       '到期月份(週別)': 'Contract', '買賣權別': 'CP', '成交數量(BorS)': 'Volume'}

            final_map = {}
            for k, v in mapping.items():
                found = [c for c in df.columns if k in c]
                if found: final_map[found[0]] = v
            df = df.rename(columns=final_map)

            required_cols = ['Symbol', 'StrikePrice', 'Price', 'CP', 'Volume', 'Contract']
            if not all(col in df.columns for col in required_cols): continue

            for col in ['Price', 'StrikePrice', 'Volume']:
                df[col] = pd.to_numeric(df[col], errors='coerce')

            mask = (df['Symbol'].str.contains('TXO', na=False)) & \
                   (df['CP'].str.extract(r'([CP])', expand=False).notna()) & \
                   (df['Volume'] >= 30) & (df['Price'] > 0)

            df_day = df[mask].copy()
            if not df_day.empty:
                df_day['TradeDate'] = raw_date_str
                df_day['File'] = f"OptionsDaily_{yyyy}_{mm}_{dd}.csv"
                df_day['Rf'] = rf
                df_day['S0'] = s0

                df_day['ContractExpiryDate'] = df_day['Contract'].apply(get_expiry_date)
                df_day['Maturity'] = (df_day['ContractExpiryDate'] - dt_object).dt.days / 365.0
                df_day['Maturity'] = df_day['Maturity'].apply(lambda x: max(x, 1/365.0) if pd.notna(x) else np.nan)

                lookup = df_day[['StrikePrice', 'Price', 'CP', 'Contract', 'Maturity']].drop_duplicates().copy()
                lookup['IV'] = lookup.apply(lambda x: find_iv(x['Price'], s0, x['StrikePrice'], x['Maturity'], rf, x['CP']), axis=1)

                df_day = df_day.merge(lookup, on=['StrikePrice', 'Price', 'CP', 'Contract', 'Maturity'], how='left')
                all_data_list.append(df_day)
        pbar.close()

    if all_data_list:
        final_df_full = pd.concat(all_data_list, ignore_index=True).dropna(subset=['IV'])

        # 定義輔助處理函數
        def get_summary_all(df_in):
            cols = ['TradeDate', 'File', 'Maturity', 'Contract', 'ContractExpiryDate', 'Rf', 'S0']
            res = df_in[cols].drop_duplicates().copy()
            res = res.rename(columns={'TradeDate': 'Date'})
            res['ContractExpiryDate'] = res['ContractExpiryDate'].apply(
                lambda x: x.strftime('%Y/%m/%d') if pd.notna(x) else ''
            )
            return res.sort_values(by=['Date', 'Contract']).reset_index(drop=True)

        def get_summary_cp(df_in):
            res = df_in.groupby('TradeDate').agg({
                'Volume': 'sum',
                'IV': ['mean', 'std', 'min', lambda x: x.quantile(0.25), 'median', lambda x: x.quantile(0.75), 'max']
            })
            res.columns = ['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']
            res.insert(1, 'IV', res['mean'])
            res.index.name = 'Date'
            return res

        for year in ['2023']:
            df_year = final_df_full[final_df_full['TradeDate'].str.startswith(year)].copy()

            if df_year.empty:
                print(f"⚠️ 找不到 {year} 年的資料，跳過存檔。")
                continue

            summary_all = get_summary_all(df_year)
            summary_call = get_summary_cp(df_year[df_year['CP'].str.contains('C', na=False)])
            summary_put = get_summary_cp(df_year[df_year['CP'].str.contains('P', na=False)])

            year_results = {
                f'TXO_Summary_All_{year}.csv': summary_all,
                f'TXO_Summary_Call_{year}.csv': summary_call,
                f'TXO_Summary_Put_{year}.csv': summary_put,
                f'TXO_Detailed_All_{year}.csv': df_year
            }

            print(f"\n--- 正在儲存 {year} 年度檔案 ---")
            for name, data in year_results.items():
                save_index = (name != f'TXO_Summary_All_{year}.csv' and name != f'TXO_Detailed_All_{year}.csv')
                data.to_csv(os.path.join(base_path, name), index=save_index, encoding='utf-8-sig')
                print(f"✅ 已儲存：{name}")

        print("\n💾 2023 年資料處理完畢！")
    else:
        print("\n❌ 未能搜集到任何符合條件的資料。")

if __name__ == "__main__":
    main()

✅ Google Drive 已在線上。
✅ 成功載入標的價格對照表：2023_S0.csv
📂 正在處理資料夾：Option_2023_Clean (共 239 個檔案)


⏳ Option_2023_Clean 進度:   0%|          | 0/239 [00:00<?, ?天/s]

KeyboardInterrupt: 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import os
import sys
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 1. 重新設定並檢查路徑 (請根據您的雲端硬碟目錄調整)
base_path = '/content/drive/MyDrive/金融資料探勘'
call_file = os.path.join(base_path, 'TXO_Summary_Call_2023.csv')
put_file = os.path.join(base_path, 'TXO_Summary_Put_2023.csv')

if not os.path.exists(call_file):
    print(f"❌ 錯誤：在 {base_path} 找不到檔案。")
    print("請確認：1. 左側有掛載 Drive 2. 資料夾名稱完全正確 3. 檔案已上傳")
    sys.exit()

# 2. 讀取並合併
df_call = pd.read_csv(call_file)
df_put = pd.read_csv(put_file)
df = pd.merge(df_call, df_put, on='Date', suffixes=('_Call', '_Put'))
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date')

# 3. 計算 MA 與 差異
for suffix in ['_Call', '_Put']:
    # IV 的 MA
    df[f'IV{suffix}_5MA'] = df[f'IV{suffix}'].rolling(window=5).mean()
    df[f'IV{suffix}_30MA'] = df[f'IV{suffix}'].rolling(window=30).mean()
    # Std 的 MA
    df[f'std{suffix}_5MA'] = df[f'std{suffix}'].rolling(window=5).mean()
    df[f'std{suffix}_30MA'] = df[f'std{suffix}'].rolling(window=30).mean()

df['IV_Diff'] = df['IV_Put'] - df['IV_Call']
df['Std_Diff'] = df['std_Put'] - df['std_Call']

# 4. 繪圖
fig, axes = plt.subplots(3, 1, figsize=(15, 22))
plt.style.use('ggplot')

# 第一張圖：IV
axes[0].plot(df['Date'], df['IV_Call'], color='#d62728', alpha=0.2, label='Call IV (Raw)')
axes[0].plot(df['Date'], df['IV_Put'], color='#2ca02c', alpha=0.2, label='Put IV (Raw)')
axes[0].plot(df['Date'], df['IV_Call_5MA'], color='#d62728', linewidth=2, label='Call IV 5MA')
axes[0].plot(df['Date'], df['IV_Put_5MA'], color='#2ca02c', linewidth=2, label='Put IV 5MA')
axes[0].plot(df['Date'], df['IV_Call_30MA'], color='#8c564b', linestyle='--', linewidth=2, label='Call IV 30MA')
axes[0].plot(df['Date'], df['IV_Put_30MA'], color='#1f77b4', linestyle='--', linewidth=2, label='Put IV 30MA')
axes[0].set_title('Implied Volatility (IV) with 5MA & 30MA', fontsize=16, fontweight='bold')
axes[0].legend(loc='upper right', ncol=2)

# 第二張圖：Std
axes[1].plot(df['Date'], df['std_Call'], color='#ff7f0e', alpha=0.2, label='Call Std (Raw)')
axes[1].plot(df['Date'], df['std_Put'], color='#1f77b4', alpha=0.2, label='Put Std (Raw)')
axes[1].plot(df['Date'], df['std_Call_5MA'], color='#ff7f0e', linewidth=2, label='Call Std 5MA')
axes[1].plot(df['Date'], df['std_Put_5MA'], color='#1f77b4', linewidth=2, label='Put Std 5MA')
axes[1].plot(df['Date'], df['std_Call_30MA'], color='#e377c2', linestyle='--', linewidth=2, label='Call Std 30MA')
axes[1].plot(df['Date'], df['std_Put_30MA'], color='#bcbd22', linestyle='--', linewidth=2, label='Put Std 30MA')
axes[1].set_title('Standard Deviation (Std) with 5MA & 30MA', fontsize=16, fontweight='bold')
axes[1].legend(loc='upper right', ncol=2)

# 第三張圖：差異長條圖
axes[2].bar(df['Date'], df['IV_Diff'], label='IV Diff (Put-Call)', color='#9467bd', alpha=0.6)
axes[2].bar(df['Date'], df['Std_Diff'], label='Std Diff (Put-Call)', color='#8c564b', alpha=0.6)
axes[2].axhline(0, color='black', linewidth=1)
axes[2].set_title('Spread Analysis (Put minus Call)', fontsize=16, fontweight='bold')
axes[2].legend(loc='upper right')

# 優化日期顯示
for ax in axes:
    ax.xaxis.set_major_locator(mdates.DayLocator([1, 15]))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
    ax.grid(True, which='major', color='gray', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

In [11]:
# ==========================================
# 第一部分：資料清洗、IV 運算與特徵統計 (最終完美版)
# ==========================================
import os
import pandas as pd
import numpy as np
import datetime
import calendar
from scipy.stats import norm
from scipy.optimize import brentq
from google.colab import drive
from tqdm import tqdm  # 進度條套件

# 1. 掛載雲端硬碟
drive.mount('/content/drive')

# ==========================================
# 選擇權核心運算函數
# ==========================================
def get_taifex_expiry_date(contract_str):
    contract_str = str(contract_str).strip()
    try:
        year = int(contract_str[:4])
        month = int(contract_str[4:6])
        c = calendar.monthcalendar(year, month)
        wednesdays = [week[2] for week in c if week[2] != 0]
        third_wed = wednesdays[2]
        return f"{year}-{month:02d}-{third_wed:02d}"
    except:
        return np.nan

def bs_price(S, K, T, r, sigma, option_type):
    if T <= 0 or sigma <= 0 or pd.isna(S):
        if option_type == 'C': return max(0, S - K)
        else: return max(0, K - S)
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if option_type == 'C': return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    elif option_type == 'P': return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
    return np.nan

def calculate_iv(row):
    S = row.get('S0', np.nan)
    K = row.get('Strike', np.nan)
    T = row.get('Maturity', 0) / 365.0
    r = row.get('rf', 0.0)
    price = row.get('Price', np.nan)
    opt_type = row.get('Put or Call', '')

    if pd.isna(S) or pd.isna(price) or T <= 0 or price <= 0: return np.nan
    try:
        return brentq(lambda sigma: bs_price(S, K, T, r, sigma, opt_type) - price, 1e-4, 2.0)
    except ValueError:
        return np.nan

def read_csv_exact(filepath, sep):
    encodings = ['utf-8-sig', 'utf-8', 'big5', 'cp950', 'utf-16']
    for enc in encodings:
        try:
            return pd.read_csv(filepath, encoding=enc, sep=sep, dtype=str)
        except Exception:
            continue
    raise ValueError(f"❌ 無法讀取檔案 {filepath}")

# ==========================================
# 主程式設定區
# ==========================================
BASE_PATH = '/content/drive/MyDrive/金融資料探勘'
S0_FILE = os.path.join(BASE_PATH, '2023_S0.csv')
SOURCE_FOLDER = os.path.join(BASE_PATH, 'Option_2023_Clean') # 確保您的資料夾叫做 Option_2023
OUTPUT_FILE = os.path.join(BASE_PATH, 'TXO_Summary_2023_Output.csv')

# ==========================================
# 步驟 1: 處理 S0 大盤資料
# ==========================================
print("📥 1. 載入 S0 資料並計算 HV...")
# 大盤資料使用 Tab (\t) 分隔
df_s0 = read_csv_exact(S0_FILE, sep='\t')
df_s0.columns = df_s0.columns.str.strip()
df_s0.rename(columns={'年月日': 'Date', '收盤價(元)': 'Close'}, inplace=True)

df_s0['Date'] = pd.to_datetime(df_s0['Date']).dt.strftime('%Y/%m/%d')
df_s0['Close'] = pd.to_numeric(df_s0['Close'], errors='coerce')
df_s0['Return'] = np.log(df_s0['Close'] / df_s0['Close'].shift(1))
df_s0['HV'] = df_s0['Return'].rolling(window=20).std() * np.sqrt(252)

s0_dict = dict(zip(df_s0['Date'], df_s0['Close']))
hv_dict = dict(zip(df_s0['Date'], df_s0['HV']))

# ==========================================
# 步驟 2: 批次處理選擇權資料夾
# ==========================================
print(f"\n📥 2. 掃描 {SOURCE_FOLDER} 資料夾中的 Option 資料...")
csv_files = [f for f in os.listdir(SOURCE_FOLDER) if f.lower().endswith('.csv')]
print(f"🔍 找到 {len(csv_files)} 個 CSV 檔案，準備合併...")

raw_data_list = []
target_cols = ['成交日期', '商品代號', '履約價格', '到期月份(週別)', '買賣權別', '成交時間', '成交價格', '成交數量(B or S)']

for filename in tqdm(csv_files, desc="讀取 CSV 進度"):
    filepath = os.path.join(SOURCE_FOLDER, filename)
    try:
        # 選擇權資料使用逗號 (,) 分隔
        df_temp = read_csv_exact(filepath, sep=',')

        # 強制清除欄位名稱的隱藏空白與換行
        df_temp.columns = df_temp.columns.str.strip().str.replace('\n', '').str.replace('\r', '')

        # 過濾垃圾列 (遇到 ---------- 這種就濾掉)
        if '成交日期' in df_temp.columns:
            df_temp = df_temp[~df_temp['成交日期'].astype(str).str.contains('-')]

        # 檢查欄位是否齊全
        if not all(col in df_temp.columns for col in target_cols):
            continue

        df_temp = df_temp[target_cols].copy()

        # 終極防呆：強制清除「商品代號」資料裡的隱藏空白
        df_temp['商品代號'] = df_temp['商品代號'].astype(str).str.strip()
        df_temp = df_temp[df_temp['商品代號'] == 'TXO'].copy()

        if not df_temp.empty:
            raw_data_list.append(df_temp)

    except Exception as e:
        pass

if not raw_data_list:
    raise ValueError("❌ 找不到任何有效的 TXO 選擇權資料，請檢查資料夾與檔案內容。")

# ==========================================
# 步驟 3: 合併並整理資料格式
# ==========================================
print("\n🧹 正在清理合併後的資料格式...")
df_opt = pd.concat(raw_data_list, ignore_index=True)

df_opt['Date'] = pd.to_datetime(df_opt['成交日期']).dt.strftime('%Y/%m/%d')
df_opt['成交時間'] = pd.to_numeric(df_opt['成交時間'], errors='coerce')
df_opt = df_opt.sort_values(by=['Date', '成交時間'])

df_opt['成交價格'] = pd.to_numeric(df_opt['成交價格'], errors='coerce')
df_opt['成交數量'] = pd.to_numeric(df_opt['成交數量(B or S)'], errors='coerce')
df_opt['履約價格'] = pd.to_numeric(df_opt['履約價格'], errors='coerce')
df_opt['買賣權別'] = df_opt['買賣權別'].str.strip().str.upper()

print("\n🔄 3. 跨檔案合併同日同合約數據 (取最後價格與總量)...")
grouped = df_opt.groupby(['Date', '到期月份(週別)', '履約價格', '買賣權別'])
consolidated_df = grouped.agg(Price=('成交價格', 'last'), Volume=('成交數量', 'sum')).reset_index()

# ==========================================
# 步驟 4: 逐日計算 IV 與特徵
# ==========================================
print("\n📊 4. 逐日計算 IV 與特徵...")
summary_results = []
total_days = consolidated_df['Date'].nunique()

for date_str, daily_df in tqdm(consolidated_df.groupby('Date'), total=total_days, desc="計算 IV 進度"):
    valid_df = daily_df[daily_df['Volume'] >= 30].copy()
    if valid_df.empty: continue

    s0_val = s0_dict.get(date_str, np.nan)
    hv_val = hv_dict.get(date_str, np.nan)

    if date_str < '2023/03/27': rf = 0.00975
    else: rf = 0.011

    valid_df['S0'] = s0_val
    valid_df['rf'] = rf
    valid_df['Strike'] = valid_df['履約價格']
    valid_df['Put or Call'] = valid_df['買賣權別']
    valid_df['Contract'] = valid_df['到期月份(週別)']
    valid_df['ContractExpiryDate_dt'] = pd.to_datetime(valid_df['Contract'].apply(get_taifex_expiry_date))
    valid_df['Maturity'] = (valid_df['ContractExpiryDate_dt'] - pd.to_datetime(date_str)).dt.days

    valid_df['IV'] = valid_df.apply(calculate_iv, axis=1)

    iv_stats = valid_df['IV'].describe()
    p_vol = valid_df[valid_df['Put or Call'] == 'P']['Volume'].sum()
    c_vol = valid_df[valid_df['Put or Call'] == 'C']['Volume'].sum()

    summary_results.append({
        'Date': date_str,
        'count': len(valid_df),
        'IV': iv_stats.get('mean', np.nan),
        'Put_Call_Ratio': p_vol / c_vol if c_vol > 0 else np.nan,
        'HV': hv_val,
        'VRP': iv_stats.get('mean', np.nan) - hv_val if pd.notna(hv_val) else np.nan,
        'Bias Ratio': (c_vol - p_vol) / (c_vol + p_vol) if (c_vol + p_vol) > 0 else np.nan
    })

print("\n💾 正在儲存最終結果...")
final_summary = pd.DataFrame(summary_results).sort_values('Date')
final_summary.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f"✨ 運算完成！摘要資料已儲存至：{OUTPUT_FILE}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📥 1. 載入 S0 資料並計算 HV...

📥 2. 掃描 /content/drive/MyDrive/金融資料探勘/Option_2023_Clean 資料夾中的 Option 資料...
🔍 找到 239 個 CSV 檔案，準備合併...


讀取 CSV 進度:   1%|▏         | 3/239 [00:09<11:54,  3.03s/it]


KeyboardInterrupt: 

In [ ]:
# ==========================================
# 第二部分：圖表視覺化繪製
# ==========================================
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import os
from google.colab import drive

# 確保已掛載
drive.mount('/content/drive')

OUTPUT_FILE = '/content/drive/MyDrive/金融資料探勘/TXO_Summary_2023_Output.csv'

if not os.path.exists(OUTPUT_FILE):
    print("❌ 找不到產出的 Summary 檔案，請先執行第一部分的程式碼。")
else:
    print("📈 正在讀取資料並繪製圖表...")
    df = pd.read_csv(OUTPUT_FILE)
    df.columns = df.columns.str.strip()
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values('Date')

    # 插值與計算 MA (5MA 與 30MA)
    cols = ['IV', 'HV', 'VRP', 'Put_Call_Ratio', 'Bias Ratio']
    df[cols] = df[cols].interpolate(method='linear', limit_direction='both')

    for col in cols:
        df[f'{col}_5MA'] = df[col].rolling(window=5, min_periods=1).mean()
        df[f'{col}_30MA'] = df[col].rolling(window=30, min_periods=1).mean()

    # 開始繪圖設定
    plt.rcParams['axes.unicode_minus'] = False
    fig, axes = plt.subplots(4, 1, figsize=(15, 24), sharex=False)

    # --- 子圖 1：IV 與 HV ---
    axes[0].plot(df['Date'], df['IV'], color='#1f77b4', alpha=0.15, label='IV Raw')
    axes[0].plot(df['Date'], df['IV_5MA'], color='#1f77b4', linewidth=2, label='IV 5MA')
    axes[0].plot(df['Date'], df['HV'], color='#ff7f0e', alpha=0.15, linestyle='--', label='HV Raw')
    axes[0].plot(df['Date'], df['HV_5MA'], color='#ff7f0e', linewidth=2, label='HV 5MA')
    axes[0].set_title('Volatility Trends: IV vs HV (5MA)', fontsize=15, fontweight='bold')
    axes[0].legend(loc='upper right', ncol=2)

    # --- 子圖 2：VRP ---
    axes[1].fill_between(df['Date'], df['VRP'], 0, where=(df['VRP'] >= 0), color='red', alpha=0.1)
    axes[1].fill_between(df['Date'], df['VRP'], 0, where=(df['VRP'] < 0), color='green', alpha=0.1)
    axes[1].plot(df['Date'], df['VRP_5MA'], color='darkred', linewidth=2, label='VRP 5MA')
    axes[1].plot(df['Date'], df['VRP_30MA'], color='black', linestyle='--', linewidth=1.5, label='VRP 30MA')
    axes[1].axhline(0, color='black', linewidth=1)
    axes[1].set_title('Volatility Risk Premium (VRP) Smoothing (30MA)', fontsize=15, fontweight='bold')
    axes[1].legend(loc='upper right')

    # --- 子圖 3：Put-Call Ratio ---
    axes[2].plot(df['Date'], df['Put_Call_Ratio'], color='purple', alpha=0.15)
    axes[2].plot(df['Date'], df['Put_Call_Ratio_5MA'], color='darkviolet', linewidth=2, label='PCR 5MA')
    axes[2].plot(df['Date'], df['Put_Call_Ratio_30MA'], color='blue', linestyle='--', linewidth=1.5, label='PCR 30MA')
    axes[2].axhline(1.0, color='red', linestyle=':', linewidth=1.2, label='Neutral Line (1.0)')
    axes[2].set_title('Sentiment: Put-Call Ratio Analysis (30MA)', fontsize=15, fontweight='bold')
    axes[2].legend(loc='upper right')

    # --- 子圖 4：Bias Ratio ---
    axes[3].plot(df['Date'], df['Bias Ratio'], color='teal', alpha=0.15)
    axes[3].plot(df['Date'], df['Bias Ratio_5MA'], color='teal', linewidth=2, label='Bias 5MA')
    axes[3].plot(df['Date'], df['Bias Ratio_30MA'], color='black', linestyle='--', linewidth=1.5, label='Bias 30MA')
    axes[3].axhline(0, color='black', linewidth=1)
    axes[3].set_title('Bias Ratio: Trend Confirmation (30MA)', fontsize=15, fontweight='bold')
    axes[3].legend(loc='upper right')

    # --- 統一格式優化 (日期標籤) ---
    for ax in axes:
        ax.xaxis.set_major_locator(mdates.DayLocator([1, 15]))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
        ax.tick_params(labelbottom=True)
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)
        ax.grid(True, which='major', color='gray', linestyle='-', alpha=0.2)
        ax.grid(True, which='minor', color='gray', linestyle=':', alpha=0.1)

    plt.tight_layout()
    plt.subplots_adjust(hspace=0.5)
    plt.show()